In [34]:
from google.colab import userdata

GITHUB_TOKEN = userdata.get("GITHUB_TOKEN")

%cd /content
!git clone https://{GITHUB_TOKEN}@github.com/Bassey-data/Auto-insurance-claim-frequency.git

%cd /content/Auto-insurance-claim-frequency
!git config user.email "basseysamuel404@gmail.com"
!git config user.name "Bassey-data"

print("Setup complete")

/content
fatal: destination path 'Auto-insurance-claim-frequency' already exists and is not an empty directory.
/content/Auto-insurance-claim-frequency
Setup complete


In [35]:
pip install statsmodels liac-arff lightgbm pyarrow shap

Import all useful modules

In [36]:

import pandas as pd
import numpy as np
import statsmodels.api as sm
from scipy.io import arff
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_poisson_deviance
import lightgbm as lgb
import shap
import matplotlib.pyplot as plt
import seaborn as sns
import os
import json
import warnings
warnings.filterwarnings("ignore")


In [37]:
# Plot styling
sns.set_theme(style="whitegrid", palette="muted")
plt.rcParams["figure.dpi"] = 110

In [38]:
def parse_arff(path):
    columns = []
    data_start = None

    with open(path, "r", encoding="utf-8", errors="replace") as f:
        lines = f.readlines()

    for i, line in enumerate(lines):
        stripped = line.strip()
        if stripped.lower().startswith("@attribute"):
            parts = stripped.split(maxsplit=2)
            columns.append(parts[1])
        elif stripped.lower().startswith("@data"):
            data_start = i + 1
            break

    data_lines = [l.strip() for l in lines[data_start:] if l.strip()]

    rows = []
    for line in data_lines:
        values = [v.strip().strip("'").strip('"') for v in line.split(",")]
        rows.append(values)

    return pd.DataFrame(rows, columns=columns)


df = parse_arff("/content/drive/MyDrive/ACQsci.arff")

print(df.head())

  IDpol ClaimNb Exposure Area VehPower VehAge DrivAge BonusMalus VehBrand  \
0     1       1      0.1    D        5      0      55         50      B12   
1     3       1     0.77    D        5      0      55         50      B12   
2     5       1     0.75    B        6      2      52         50      B12   
3    10       1     0.09    B        7      0      46         50      B12   
4    11       1     0.84    B        7      0      46         50      B12   

    VehGas Density Region  
0  Regular    1217    R82  
1  Regular    1217    R82  
2   Diesel      54    R22  
3   Diesel      76    R72  
4   Diesel      76    R72  


In [39]:
df.dtypes

,0
IDpol,object
ClaimNb,object
Exposure,object
Area,object
VehPower,object
VehAge,object
DrivAge,object
BonusMalus,object
VehBrand,object
VehGas,object


fix numeric and categorical columns data types

In [40]:
# used an f string for readable output
print(f"Shape: {df.shape}")
print(f"\nMissing values:\n{df.isna().sum()}")
print(f"\nDuplicate rows: {df.duplicated().sum()}")

Shape: (678013, 12)

Missing values:
IDpol         0
ClaimNb       0
Exposure      0
Area          0
VehPower      0
VehAge        0
DrivAge       0
BonusMalus    0
VehBrand      0
VehGas        0
Density       0
Region        0
dtype: int64

Duplicate rows: 0


In [41]:
df_converted = df.apply(pd.to_numeric, errors="ignore")

for col in df_converted.select_dtypes(include="object").columns:
    df_converted[col] = df_converted[col].astype("category")

df = df_converted
print(df.dtypes)

IDpol          float64
ClaimNb          int64
Exposure       float64
Area          category
VehPower         int64
VehAge           int64
DrivAge          int64
BonusMalus       int64
VehBrand      category
VehGas        category
Density          int64
Region        category
dtype: object


In [42]:
print(f"Shape: {df.shape}")
print(f"\nMissing values:\n{df.isna().sum()}")
print(f"\nDuplicate rows: {df.duplicated().sum()}")

Shape: (678013, 12)

Missing values:
IDpol         0
ClaimNb       0
Exposure      0
Area          0
VehPower      0
VehAge        0
DrivAge       0
BonusMalus    0
VehBrand      0
VehGas        0
Density       0
Region        0
dtype: int64

Duplicate rows: 0


In [43]:
os.makedirs("data/processed", exist_ok=True)

df.to_parquet("data/processed/freMTPL2freq.parquet", index=False)

print("Saved successfully")
print(f"Parquet size: {os.path.getsize('data/processed/freMTPL2freq.parquet') / 1e6:.1f} MB")

Saved successfully
Parquet size: 7.5 MB


In [47]:
print(os.path.exists("/content/Auto-insurance-claim-frequency/data/processed/freMTPL2freq.parquet"))


True


In [49]:
%cd /content/Auto-insurance-claim-frequency
!git add .
!git commit -m "Add data preparation notebook and processed parquet file"
!git push

/content/Auto-insurance-claim-frequency
On branch main
Your branch is up to date with 'origin/main'.

nothing to commit, working tree clean
Everything up-to-date
